In [ ]:
import os
from dotenv import load_dotenv

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import JinaEmbeddings


In [10]:
load_dotenv()

True

In [11]:
groq_key = os.getenv("GROQ_API_KEY")
jina_key = os.getenv("JINA_API_KEY")

print("Environment variable loaded :)")

Environment variable loaded :)


In [12]:
#Loading our data file
DATA_DIR = os.path.join(os.getcwd(), "data","hr_policy.txt")

In [13]:
#data ingestion
loader = TextLoader(DATA_DIR,encoding="utf-8")
documents = loader.load()

print(documents)


[Document(metadata={'source': 'd:\\langchian project\\RagHRPolicy\\data\\hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)\n\n1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.\n\n2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM to 4 PM.\n\n3. PROBATION PERIOD\nAll new employees undergo a probation period of 3 months from th

In [17]:
print(documents[0].metadata)

{'source': 'd:\\langchian project\\RagHRPolicy\\data\\hr_policy.txt'}


In [19]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
)

chunks = text_splitter.split_documents(documents)
print(chunks)

[Document(metadata={'source': 'd:\\langchian project\\RagHRPolicy\\data\\hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)'), Document(metadata={'source': 'd:\\langchian project\\RagHRPolicy\\data\\hr_policy.txt'}, page_content='1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.'), Document(metadata={'source': 'd:\\langchian project\\RagHRPolicy\\data\\hr_policy.txt'}, page_content='2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require writte

In [22]:
len(chunks)
print(chunks[1])

page_content='1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.' metadata={'source': 'd:\\langchian project\\RagHRPolicy\\data\\hr_policy.txt'}


In [23]:
##Embed our data using Jina Embeddings
from langchain_community.embeddings import JinaEmbeddings

embeddings_model = JinaEmbeddings(model_name="jina-embeddings-v2-base-en")
print("embeddings model loaded :)",embeddings_model.model_name)


embeddings model loaded :) jina-embeddings-v2-base-en


In [25]:
#Store data in vector database using Groq
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(documents=chunks, embedding=embeddings_model)
print("vector store created :)",vector_store.index.ntotal)

vector store created :) 9


In [26]:
test_query = "How many sick leaves employees get?"
#similarity search
similar_docs = vector_store.similarity_search(test_query, k=3)
print("similar docs :)",similar_docs)
for i,match in enumerate(similar_docs):
    print(f"Match {i+1}:")
    print(f"Content: {match.page_content}")
    print(f"Metadata: {match.metadata}")
    print() 

similar docs :) [Document(id='1ce782d9-6d88-4322-91d1-efdfb2f7599d', metadata={'source': 'd:\\langchian project\\RagHRPolicy\\data\\hr_policy.txt'}, page_content='1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.'), Document(id='9dc18bc1-dffc-461b-ad8f-13bb3e75811d', metadata={'source': 'd:\\langchian project\\RagHRPolicy\\data\\hr_policy.txt'}, page_content='7. HOLIDAYS\nThe company observes 12 public holidays every year, as per the official holiday calendar\npublished by HR at the start of each year.\nEmployees working on a public holiday are eligible for compensatory leave.'), Docume